In [141]:
# Import modules

from IPython.display import Image
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.animation import FuncAnimation
import pandas as pd
import numpy as np

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
# Enable interactive plot
%matplotlib notebook
%matplotlib notebook

import subprocess
import sys

# Import PySwarms
import pyswarms as ps
from pyswarms.utils.functions import single_obj as fx
from pyswarms.utils.plotters import (plot_cost_history, plot_contour, plot_surface)
from pyswarms.utils.plotters.formatters import Mesher

global particle_number
particle_number=0

from math import floor

import random

particles = 1
iterations = 1
nb_test_0 = 20
nb_test_2 = 120

indexes_0 = []
indexes_2 = []

taille = len(push_data)-1
for i in range(nb_test_0):
    indexes_0.append(int((i+1)*(taille/(nb_test_0+1))))

taille = len(push_data2)
for i in range(nb_test_2):
    indexes_2.append(int((i+1)*(taille/(nb_test_2+1))))

print(indexes_0)  
print(indexes_2)    
  


[3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 34, 37, 40, 43, 46, 49, 52, 55, 58, 61]
[1, 3, 4, 6, 7, 9, 10, 12, 13, 15, 16, 18, 19, 21, 22, 24, 25, 27, 29, 30, 32, 33, 35, 36, 38, 39, 41, 42, 44, 45, 47, 48, 50, 51, 53, 55, 56, 58, 59, 61, 62, 64, 65, 67, 68, 70, 71, 73, 74, 76, 77, 79, 81, 82, 84, 85, 87, 88, 90, 91, 93, 94, 96, 97, 99, 100, 102, 103, 105, 107, 108, 110, 111, 113, 114, 116, 117, 119, 120, 122, 123, 125, 126, 128, 129, 131, 133, 134, 136, 137, 139, 140, 142, 143, 145, 146, 148, 149, 151, 152, 154, 155, 157, 159, 160, 162, 163, 165, 166, 168, 169, 171, 172, 174, 175, 177, 178, 180, 181, 183]


## Load the goal data

In [72]:
push_data = pd.read_csv('PushData.csv',sep=";")[['MaxForce', 'PerturbationDuration','Final_CoM_y','angle','StepIndex','Impulse']]
push_data = push_data.loc[push_data['angle']==90]
push_data2 = push_data.loc[push_data['StepIndex']==2]
push_data2=push_data2[['MaxForce', 'PerturbationDuration','Final_CoM_y','Impulse']]
push_data = push_data.loc[push_data['StepIndex']==0]
push_data=push_data[['MaxForce', 'PerturbationDuration','Final_CoM_y','Impulse']]

push_data = push_data.sort_values(by=['Impulse'])
push_data2 = push_data2.sort_values(by=['Impulse'])


print(len(push_data))
print(len(push_data2))


push_data.head()
#push_data2.head()

result=[-200,-200,0,-300, -250, -250, -20, -20, -350, -10, -20,   -10,-10,0,-10, -10, -10,  -5, -1, -20, -4, 0]

66
185


## Fonction evaluant la stabilité du CoM en fonction des poids kd,kp

In [131]:
def weight_impact_function(param,force,time,index):
   
    #kds = kd1,kd2,kd3,kd4,kd5,kd6,kd7,kd8,kd9,kd10,kd11,kd12,kd13,kd14,kd15,kd16,kd17,kd18
    #kps = kp1,kp2,kp3,kp4,kp5,kp6,kp7,kp8,kp9,kp10,kp11,kp12,kp13,kp14,kp15,kp16,kp17,kp18
    
    
    
    param = [param[0],param[1],param[2],param[3],param[4],param[5],param[5],param[3],param[4],param[6],param[7],
             param[6],param[7],param[8],param[8],param[9],param[10],param[9],param[10],
             
             param[11],param[12],param[13],param[14],param[15],param[16],param[16],param[14],param[15],param[17],
             param[18],param[17],param[18],param[19],param[19],param[20],param[21],param[20],param[21]
            ]
    
    #print(param)
    for i in {2,21,35,37}:
        param[i]=0
    
    
    f = open("input"+str(index)+".txt", "w")
    f.write(('\n'.join(str(param[i]) + " "+ str(param[i+19]) for i in range(19))).replace("[","").replace("]",""))
    f.write("\n" + str(force) + " " + str(time) +"\n")
    f.close()

    #subprocess.check_call(['../../bin/App_BulletExampleBrowser_vs2010_x64_debug.exe'])
    
    
    
    donnee_1 = pd.read_csv("output.txt", sep=" ")
    result = donnee_1["Z"].iat[-1]
    return result


def all_particles(input_param):
    result =[0] * particles
    
    for i in range(particles):
        print(input_param[i][0])
    
    
    for j in range(nb_test_0):
        current_test = push_data.iloc[indexes_0[j]].to_numpy()
        #current_test = push_data.iloc[j].to_numpy()
        #print(current_test)
        
        list_program = []
        for i in range(particles):
            temp = -(abs(current_test[2])+weight_impact_function(input_param[i],current_test[0],current_test[1],i))
            #result[i] = result[i] + temp
            #print("Diff "+ str(j) + "/" + str(len(push_data)) +" : " + str(-abs(current_test[2])) + " - " + str(temp+abs(current_test[2])) + " -> " + str(temp) +"\n")
            list_program.append('../../bin/App_BulletExampleBrowser_vs2010_x64_release'+str(i)+'.exe --cl_device=1')
            
        processes = [subprocess.Popen(program) for program in list_program]
        # wait
        for process in processes:
            process.wait()
        
        for i in range(particles):
            dataa.append([current_test[0],current_test[1],current_test[2],temp,current_test[3]])
            result[i] = result[i] + analyse(current_test[3],0,i) / (nb_test_0+nb_test_2)

    
    
    
    for j in range(nb_test_2):
        current_test = push_data2.iloc[indexes_2[j]].to_numpy()
        #current_test = push_data.iloc[j].to_numpy()
        print(current_test)
        
        list_program = []
        for i in range(particles):
            temp = -(abs(current_test[2])+weight_impact_function(input_param[i],current_test[0],current_test[1],i))
            #result[i] = result[i] + temp
            #print("Diff "+ str(j) + "/" + str(len(push_data)) +" : " + str(-abs(current_test[2])) + " - " + str(temp+abs(current_test[2])) + " -> " + str(temp) +"\n")
            list_program.append('../../bin/App_BulletExampleBrowser_vs2010_x64_release'+str(i)+'.exe')
            
        processes = [subprocess.Popen(program) for program in list_program]
        # wait
        for process in processes:
            process.wait()
        
        for i in range(particles):
            dataa.append([current_test[0],current_test[1],current_test[2],temp,current_test[3]])
            result[i] = result[i] + analyse(current_test[3],0,i) / (nb_test_0+nb_test_2)
         
    print("Resultat : " + str(result))
    result = result 
    return result
        
        

## Lancer la fonction une fois

In [ ]:
kds = -1200, 0, -900, -900, -20, -20, -900, -900, -20, -20, -20, -20, -1200, -1200, -20, -10, -20, -10
kps = -10,0,-10, -10, -1, -1, -10, -10, -1.5, -1.5, -1.5, -1.5, -10, -10, -1, 0, -1, 0

result = weight_impact_function(-1200, 0, -900, -900, -20, -20, -900, -900, -20, -20, -20, -20, -1200, -1200, -20, -10, -20, -10,-10,0,-10, -10, -1, -1, -10, -10, -1.5, -1.5, -1.5, -1.5, -10, -10, -1, 0, -1, 0, 36,0.55)
print(result)
##donnee_1.head()

## Optimiser

In [134]:
dataa=[]

kwargs = { "kd2":0, "kd3":-900, "kd4":-900, "kd5":-20, "kd6":-20, "kd7":-900, "kd8":-900, "kd9":-20, "kd10":-20, "kd11":-20, "kd12":-20, "kd13":-1200, "kd14":-1200, "kd15":-20, "kd16":-10, "kd17":-20, "kd18":-10,"kp2":0,"kp3":-10, "kp4":-10, "kp5":-1, "kp6":-1, "kp7":-10, "kp8":-10, "kp9":-1.5, "kp10":-1.5, "kp11":-1.5, "kp12":-1.5, "kp13":-20, "kp14":-20, "kp15":-1, "kp16":0, "kp17":-1, "kp18":0,"force":36,"time":0.55}
kwargs={}
max_bound = 0.5*np.array(result)
min_bound = 1*np.array(result)
print(result)
bounds = (min_bound, max_bound)
options = {'c1': 0.4, 'c2': 0.4, 'w':0.9,'max_velocity_rate':10000000,'adaptive':True}
np_vers = np.array([result]*particles).astype(np.float32)
optimizer = ps.single.GlobalBestPSO(n_particles=particles, dimensions=22, options=options, bounds=bounds)
cost, pos = optimizer.optimize(all_particles, **kwargs,iters=iterations)

memoire=pos
f = open("resultat.txt", "w")
for i in range(len(memoire)):
    f.write((str(memoire[i]) + " "))
    f.write("\n")
f.close()

2023-03-27 14:30:45,612 - pyswarms.single.global_best - INFO - Optimize for 100 iters with {'c1': 0.4, 'c2': 0.4, 'w': 0.9, 'max_velocity_rate': 10000000, 'adaptive': True}


[-200, -200, 0, -300, -250, -250, -20, -20, -350, -10, -20, -10, -10, 0, -10, -10, -10, -5, -1, -20, -4, 0]


pyswarms.single.global_best:   0%|                                                                               |0/100

-161.0084528207156
-187.87350635796275
-104.01035151637815
-126.68192177280055
-188.83163744827723
-104.19817801516545
-151.99081935736223
-134.33994066870554
-159.4835277590203
-130.10679974124645
-135.95815387103892
-132.44383808569188
-142.76113187246543
-187.77761409162937
-128.34460811100584
-100.13045330998285
-129.58451851287444
-164.0540942225202
-159.67459046722973
-130.72225706466543
0CLEAR!

1CLEAR!

2CLEAR!

3CLEAR!

4CLEAR!

5CLEAR!

6CLEAR!

7CLEAR!

8CLEAR!

9CLEAR!

10CLEAR!

11CLEAR!

12CLEAR!

13CLEAR!

14CLEAR!

15CLEAR!

16CLEAR!

17CLEAR!

18CLEAR!

19CLEAR!

[78.456  0.66   0.455 23.386]
0CLEAR!

1CLEAR!

2CLEAR!

3CLEAR!

4CLEAR!

5CLEAR!

6CLEAR!

7CLEAR!

8CLEAR!

9CLEAR!

10CLEAR!

11CLEAR!

12CLEAR!

13CLEAR!

15CLEAR!

16CLEAR!

17CLEAR!

18CLEAR!

19CLEAR!

[77.103  0.875  0.693 35.292]
0CLEAR!

2CLEAR!

3CLEAR!

4CLEAR!

5CLEAR!

6CLEAR!

7CLEAR!

8CLEAR!

10CLEAR!

11CLEAR!

14CLEAR!

15CLEAR!



pyswarms.single.global_best:   0%|                                                                |0/100, best_cost=309c:\users\ajensen\appdata\local\programs\python\python38\lib\site-packages\pyswarms\backend\handlers.py:387: RuntimeWarning: invalid value encountered in remainder
  new_pos[greater_than_bound] = lb[greater_than_bound] + np.mod(
pyswarms.single.global_best:   1%|▋                                                               |1/100, best_cost=309

...

18CLEAR!

19CLEAR!

Resultat : [322.8651755005665, 391.57037638451584, 413.76865488693267, 337.83102465335014, 435.06628501029957, 292.33449864155034, 312.0728680290671, 323.06020226689986, 323.62365687561646, 329.8020715242002, 336.2917088610836, 435.2852843637165, 363.0312590546994, 290.50351446731656, 329.0350639448168, 354.33842495590005, 376.66159390928414, 337.7279907836835, 306.0598621550665, 348.6246509821334]
-149.59337680429852
-150.14407877837917
-154.34829555904426
-158.82496812421792
-164.01573971850752
-155.3091073031595
-154.12874386380807
-156.2929794266343
-169.43768552944817
-153.05284967413664
-150.1632935286553
-153.6079594296662
-155.62341412293037
-159.91781613456885
-153.39904656933098
-151.6846378855344
-142.01695583786446
-166.15640259004104
-157.14332541696626
-154.1834677936992
0CLEAR!

1CLEAR!

2CLEAR!

3CLEAR!

4CLEAR!

5CLEAR!

6CLEAR!

7CLEAR!

8CLEAR!

9CLEAR!

10CLEAR!

11CLEAR!

12CLEAR!

13CLEAR!

14CLEAR!

15CLEAR!

16CLEAR!

17CLEAR!

18CLEAR!

19CL

pyswarms.single.global_best:  99%|██████████████████████████████████████████████████████████████▎|99/100, best_cost=231

17CLEAR!

18CLEAR!

19CLEAR!

Resultat : [326.91174838453344, 372.0707910925165, 427.9587067211005, 338.7287205610005, 385.4048079321832, 292.84399719794976, 290.6875160466663, 307.63163557758384, 348.81051853824994, 382.3652293232001, 332.1912113165667, 322.3300638271669, 319.32832632700024, 308.35682717186705, 320.04711639739924, 283.7518711768166, 337.49827549664974, 349.0326834656335, 367.43878499715004, 352.19472173951647]
-146.76592267390478
-151.06456310093682
-153.8904957180999
-157.7420852385864
-159.66875312058542
-147.42725814900945
-153.66245712541385
-155.37943222415737
-168.4395740779347
-152.3560108194275
-148.83571404421997
-153.87775421119562
-155.4479110697376
-158.06493017392182
-154.09634078067515
-151.55752925735888
-144.24614301396835
-170.6755251862558
-155.27680050012333
-154.63522193996627
0CLEAR!

1CLEAR!

2CLEAR!

3CLEAR!

4CLEAR!

5CLEAR!

6CLEAR!

7CLEAR!

8CLEAR!

9CLEAR!

10CLEAR!

11CLEAR!

12CLEAR!

13CLEAR!

14CLEAR!

15CLEAR!

16CLEAR!

17CLEAR!

18CL

pyswarms.single.global_best: 100%|██████████████████████████████████████████████████████████████|100/100, best_cost=231
2023-03-27 22:02:05,298 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 231.296803006667, best pos: [-154.306 -167.928      nan -255.95  -199.63  -183.119  -13.24   -10.975
 -318.454   -6.335  -18.488   -6.912   -5.256      nan   -5.663   -7.448
   -7.99    -3.83    -0.688  -15.311   -3.578      nan]


18CLEAR!

19CLEAR!

Resultat : [347.1667968127832, 347.11834959733346, 407.67590852828346, 351.0130259174831, 326.2245003105002, 285.10631223671624, 321.78642633711684, 294.90160641346665, 336.8199777274163, 345.0723683681331, 350.0448604062168, 278.14087320031695, 258.55305160371614, 293.0129602622335, 303.1391639163829, 358.3694882365162, 335.4236254925671, 307.3433530269498, 289.5259413014004, 370.7753648014998]


## Visualiser l'optimisation

In [145]:
np.set_printoptions(suppress=True)
np.set_printoptions(precision=3)
#print(pos)
#print(len(dataa))
x=[]
y=[]
y2=[]

donnee_1 = pd.read_csv('output.txt', sep=" ")

plt.xlabel('Applied impulse')
plt.ylabel('Total displacement')
plt.title('Comparison of simulation and real data (simulation in orange)')
    
def get_cmap(n, name='hsv'):
    '''Returns a function that maps each index in 0, 1, ..., n-1 to a distinct 
    RGB color; the keyword argument name must be a standard mpl colormap name.'''
    return plt.cm.get_cmap(name, n)

case=1
if(case==1):
    
    plt.xlabel('Iteration')
    plt.ylabel('Cost function')
    plt.title('Cost history thoughout optimisation')

    
    #print(optimizer.cost_history)

    for i in range(len(optimizer.cost_history)):
        x.append(i)
        y.append(optimizer.cost_history[i])
    plt.plot(x,y, 'ro', linestyle='-')
else :
    if (case==2):
        #plt.ylim(-0.3,1.3)
        #plt.xlim(-0.1,1.3)
        #plt.xlim(10,60)

        #plt.plot([-0.1, 1000], [0.15, 0.15], 'r-.', lw=2)
        #plt.plot([-0.1, 1000], [-0.15, -0.15], 'r-.', lw=2)
        #plt.plot([-100, -0.1], [1000, -0.1], 'r-', lw=2)

        print(optimizer.cost_history)

        for i in range(len(dataa)):
            x.append(dataa[i][4])
            y.append(dataa[i][3]+abs(dataa[i][2]))
            #y.append(dataa[i][3])
            #y.append(0.95*dataa[i][2]+(dataa[i][3]-0.37))
            y2.append(dataa[i][2])
        plt.plot(x,y, 'ro', linestyle='None')
        plt.plot(x,y2, 'bo', linestyle='None')

        coef = np.polyfit(x,y,1)
        poly1d_fn = np.poly1d(coef) 
        coef_data = np.polyfit(x,y2,1)
        poly1d_fn_data = np.poly1d(coef_data) 
        # poly1d_fn is now a function which takes in x and returns an estimate for y
        plt.plot(x,y,'o', x, poly1d_fn(x), 'r-')

        plt.plot(x,y,'o', x, poly1d_fn_data(x), 'b-')

        #plt.plot([0, 2], [0, 2], 'g-', lw=2)

        #plot_cost_history(cost_history=optimizer.cost_history)
        print(str(np.sum(y)/len(y)))
    else:
        if(case==3):
            cmap = get_cmap(18)

            for j in range(0,1):
                x=[]
                y=[]
                for i in range(len(donnee_1.iloc[:, j])):
                    x.append(i)
                    y.append(donnee_1.iloc[:, j].iloc[i])
                plt.plot(x,y, c=cmap(j), linestyle='-')
                
                print(100*sum(y)/200)
                
            for j in range(1,2):
                x=[]
                y=[]
                for i in range(len(donnee_1.iloc[:, j])):
                    x.append(i)
                    y.append(donnee_1.iloc[:, j].iloc[i])
                plt.plot(x,y, c=cmap(j), linestyle='-')
                
                print(0.01*sum(y)/200)
                
                
if(case==4):
    origin=[[-200,-200,0,-300, -250, -250, -20, -20, -350, -10, -20,   -10,-10,0,-10, -10, -10,  -5, -1, -20, -4, 0]] 
    temp = particles
    particles=1
    val_origin = all_particles(origin)
    donnee_ori = pd.read_csv('output.txt', sep=" ")
    val_result = all_particles([result])
    donnee_res = pd.read_csv('output.txt', sep=" ")  
    particles=temp
    
if(case==5):
    
    plt.xlabel('Time')
    plt.ylabel('Cost function')
    plt.title('Comparison of pre(blue) and post(red) optimisation')

    print(origin)
    print(result)
    scan=range(1,2)
    
    
    cmap = get_cmap(18)
    

    x=[0]*len(donnee_ori.iloc[:, j])
    y=[0]*len(donnee_ori.iloc[:, j])
    for j in scan:
            for i in range(len(donnee_ori.iloc[:, j])):
                x[i] = i
                y[i] += donnee_ori.iloc[:, j].iloc[i]
    plt.plot(x,y, 'b', linestyle='-')
    
    

    x=[0]*len(donnee_res.iloc[:, j])
    y=[0]*len(donnee_res.iloc[:, j])
    for j in scan:
            for i in range(len(donnee_res.iloc[:, j])):
                x[i] = i
                y[i] += donnee_res.iloc[:, j].iloc[i]/len(donnee_res.iloc[:, j])
    plt.plot(x,y, 'r', linestyle='-')

    print(val_origin)
    print("Valeur de base : " + str(val_origin) + " --->" + str(val_result) + " ( -"+ str(100*(val_origin[0]-val_result[0])/val_origin[0]) + " % )")

    
if(case==6):
    x=[0]*20
    y=[0]*20
    print(origin)
    print(result)
    
    for i in range(0,20):
        x[i] = i
        y[i] = (result[i]-0.5*origin[0][i])/(origin[0][i])
    #plt.plot(x,origin[0], 'r', linestyle='-')
    plt.plot(x,y, 'b', linestyle='-')
plt.show()

## Sauvegarde des données

In [135]:
print(memoire)

memoire = dataa
result = pos
temp=result
result[np.isnan(result)] = 0
np.set_printoptions(suppress=True)
np.set_printoptions(precision=3)
for i in range(int(len(result)/2)) :
    print(str(result[i])+ " " +str(result[int(i+len(result)/2)]))

val_original = np.array([-500,0,-300, -500, -500, -10, -10, -1000, -50, -50,   -100,0,-10, -20, -20,  -1, -1, -20, -5, 0]).astype(np.float32)
print((result - val_original*0.8)/( val_original*0.4))
print("")
print(val_original*1.2-result)
f = open("resultat.txt", "w")
for i in range(len(memoire)):
    for j in range(len(memoire[0])):
        f.write((str(memoire[i][j]) + " "))
    f.write("\n")
f.close()

[-154.306 -167.928      nan -255.95  -199.63  -183.119  -13.24   -10.975
 -318.454   -6.335  -18.488   -6.912   -5.256      nan   -5.663   -7.448
   -7.99    -3.83    -0.688  -15.311   -3.578      nan]
-154.30561625375302 -6.912248960902451
-167.92753399601594 -5.256226280396702
0.0 0.0
-255.94980083405892 -5.662780617646219
-199.6299901045049 -7.447979113954261
-183.11858566172643 -7.9895545374333095
-13.240285083985926 -3.8298196434918395
-10.975403901809992 -0.6883438001458668
-318.45398489746776 -15.310907568420085
-6.334526819971689 -3.577949222165904
-18.488481498201523 0.0


ValueError: operands could not be broadcast together with shapes (22,) (20,) 

In [28]:
donnee_1 = pd.read_csv('output.txt', sep=" ")
donnee_1.head()


,q,torque,X,Z,vX,vZ,pasEffectue,Unnamed: 7
0,0.00,0.00,0.000000,-0.002931,0.000000,-0.002931,0,NaN
1,0.02,2.88,0.000000,-0.002931,0.000000,-0.002931,0,NaN
2,0.02,2.88,0.000000,-0.002931,0.000000,-0.002931,0,NaN
3,0.02,2.88,0.000000,-0.002931,0.000000,-0.002931,0,NaN
4,0.02,2.88,-0.000085,-0.003738,-0.005203,-0.052137,0,NaN


## Resultat ?

In [365]:
print(optimizer.pos_history[1][2])
result = optimizer.pos_history[1][2]

[-489.115      nan -307.8   -474.099 -496.728  -10.337  -11.807 -978.295
  -52.907  -43.981 -106.011      nan  -10.75   -16.522  -20.065   -1.082
   -1.187  -20.082   -4.452      nan]


In [214]:


print(len(push_data2))
x=[]
y=[]
plt.xlabel('Applied impulse')
plt.ylabel('Total displacement')
plt.title('Comparison of simulation and real data (simulation in orange)')
    
    
#plt.ylim(-0.3,1.3)
#plt.xlim(-0.1,1.3)
plt.xlim(10,60)
 
#plt.plot([-0.1, 1000], [0.15, 0.15], 'r-.', lw=2)
#plt.plot([-0.1, 1000], [-0.15, -0.15], 'r-.', lw=2)
#plt.plot([-100, -0.1], [1000, -0.1], 'r-', lw=2)



for i in range(len(push_data2)):
    x.append(push_data2.iloc[i][3])
    y.append(push_data2.iloc[i][2])
plt.plot(x,y, 'ro', linestyle='None')

coef = np.polyfit(x,y,1)
poly1d_fn = np.poly1d(coef) 
# poly1d_fn is now a function which takes in x and returns an estimate for y
plt.plot(x,y,'o', x, poly1d_fn(x), 'r-')
print(coef)

plt.plot([0, 2], [0, 2], 'g-', lw=2)

#plot_cost_history(cost_history=optimizer.cost_history)
print(str(np.sum(y)/len(y)))
#plt.show()

185
[0.014 0.069]
0.4914289440207285


In [112]:
#import matplotlib.pyplot as plt2
x=[]
y=[]
for i in range(len(push_data2)):
    x.append(push_data2.iloc[i][3])
    y.append(push_data2.iloc[i][2])

coef = np.polyfit(x,y,1)
poly1d_fn_1 = np.poly1d(coef) 

x=[]
y=[]
for i in range(len(push_data)):
    x.append(push_data2.iloc[i][3])
    y.append(push_data2.iloc[i][2])

coef = np.polyfit(x,y,1)
poly1d_fn_2 = np.poly1d(coef) 

def compute_jerk(velocities):
    
    x=[]
    y=[] #vélocités
    y2=[] #acceleration
    y3=[] #jerk
    
    x.append(0)
    y.append(0)
    y2.append(0)
    y3.append(0)
    for i in range(len(donnee_1)):
        x.append(i)
        y.append(velocities[i])
        y2.append((y[i]-y[i-1])*10)
        y3.append((y2[i]-y2[i-1])*10)
    return sum(map(abs, y3))

def compute_vel(velocities):
    
    x=[]
    y=[] #vélocités
    y2=[] #acceleration
    
    x.append(0)
    y.append(0)
    y2.append(0)
    for i in range(len(donnee_1)):
        x.append(i)
        y.append(velocities[i])
        y2.append((y[i]-y[i-1])*10)
    return sum(map(abs, y2))

def analyse(impulse,pasEntree,index):
    donnee_1 = pd.read_csv('output'+str(index)+'.txt', sep=" ")
    
    result = 0
    for j in range(2):
        for i in range(len(donnee_1.iloc[:, j])):
               result += donnee_1.iloc[:, j].iloc[i]
                
    result=0
    for j in range(0,1):
        x=[]
        y=[]
        for i in range(len(donnee_1.iloc[:, j])):
            x.append(i)
            y.append(donnee_1.iloc[:, j].iloc[i])
             
        if((100*sum(y)/len(donnee_1.iloc[:, j]))>70):
            result += 1000*sum(y)/len(donnee_1.iloc[:, j])
        else :
            print(str(index)+"CLEAR!\n")
            
    result=0           
    for j in range(1,2):
        x=[]
        y=[]
        for i in range(len(donnee_1.iloc[:, j])):
            x.append(i)
            y.append(donnee_1.iloc[:, j].iloc[i])
                
        result += 0.01*sum(y)/200
              
    return result